In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.drawing.image import Image


# ==========================
# BACA DATA
# ==========================

file_csv = "data_logger.csv"

data = pd.read_csv(file_csv)

loggers = data.columns[1:]


# ==========================
# HITUNG F0 SEMUA LOGGER
# ==========================

hasil_f0 = []

for logger in loggers:

    suhu_C = (
        (data[logger] - 32)
        * 5 / 9
    )

    LR = 10 ** (
        (suhu_C - 121.1)
        / 10
    )

    letalitas = (
        (
            data["Waktu"]
            - data["Waktu"].shift(1)
        )
        *
        (
            (
                LR
                + LR.shift(1)
            ) / 2
        )
    ).fillna(0)

    F0 = letalitas.sum()

    hasil_f0.append([
        logger,
        F0,
        data[logger].max(),
        data[logger].min()
    ])



# ==========================
# TABEL F0 LOGGER
# ==========================

summary_logger = pd.DataFrame(
    hasil_f0,
    columns=[
        "Logger",
        "F0",
        "Tmax",
        "Tmin"
    ]
)

summary_logger = summary_logger.sort_values(
    "F0"
)


# ==========================
# COLD SPOT
# ==========================

cold_logger = summary_logger.iloc[0]["Logger"]

cold_f0 = summary_logger.iloc[0]["F0"]


# ==========================
# HOT SPOT
# ==========================

hot_logger = summary_logger.iloc[-1]["Logger"]

hot_f0 = summary_logger.iloc[-1]["F0"]


# ==========================
# DATA COLD SPOT
# ==========================

cold_data = pd.DataFrame()

cold_data["Waktu"] = data["Waktu"]

cold_data["suhu_F"] = data[cold_logger]

cold_data["suhu_C"] = (
    (cold_data["suhu_F"] - 32)
    * 5 / 9
)

cold_data["LR"] = (
    10 ** (
        (
            cold_data["suhu_C"]
            - 121.1
        )
        / 10
    )
)

cold_data["letalitas"] = (
    (
        cold_data["Waktu"]
        - cold_data["Waktu"].shift(1)
    )
    *
    (
        (
            cold_data["LR"]
            + cold_data["LR"].shift(1)
        )
        / 2
    )
).fillna(0)

cold_data["F0_kumulatif"] = (
    cold_data["letalitas"]
    .cumsum()
)


# ==========================
# SUMMARY
# ==========================

summary = pd.DataFrame({

    "Parameter":[

        "Jumlah Logger",

        "Cold Spot",

        "F0 Cold Spot",

        "Hot Spot",

        "F0 Hot Spot",

        "F0 Total"

    ],

    "Nilai":[

        len(loggers),

        cold_logger,

        round(cold_f0,2),

        hot_logger,

        round(hot_f0,2),

        round(
            cold_data["F0_kumulatif"]
            .iloc[-1],
            2
        )

    ]
})


# ==========================
# GRAFIK SUHU
# ==========================

plt.figure(figsize=(10,5))

plt.plot(
    cold_data["Waktu"],
    cold_data["suhu_F"]
)

plt.title(
    f"Cold Spot {cold_logger}"
)

plt.xlabel("Waktu")

plt.ylabel("Suhu F")

plt.grid()

plt.savefig(
    "kurva_suhu.png"
)

plt.close()


# ==========================
# GRAFIK F0
# ==========================

plt.figure(figsize=(10,5))

plt.plot(
    cold_data["Waktu"],
    cold_data["F0_kumulatif"]
)

plt.title(
    "F0 Kumulatif"
)

plt.xlabel("Waktu")

plt.ylabel("F0")

plt.grid()

plt.savefig(
    "kurva_f0.png"
)

plt.close()


# ==========================
# EXCEL
# ==========================

excel_file = "Laporan_Retort.xlsx"

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    summary_logger.to_excel(
        writer,
        sheet_name="F0 Logger",
        index=False
    )

    cold_data.to_excel(
        writer,
        sheet_name="Data Retort",
        index=False
    )


# ==========================
# TAMBAH GAMBAR
# ==========================

wb = load_workbook(
    excel_file
)

ws = wb.create_sheet(
    "Kurva"
)

ws.add_image(
    Image("kurva_suhu.png"),
    "A1"
)

ws.add_image(
    Image("kurva_f0.png"),
    "A30"
)

wb.save(
    excel_file
)

print(
    "Laporan selesai dibuat"
)

Laporan selesai dibuat


In [ ]:
plt.figure(figsize=(12, 7))
for logger in loggers:
    plt.plot(data["Waktu"], data[logger], label=logger)

plt.title("Grafik Suhu Lengkap Semua Logger")
plt.xlabel("Waktu")
plt.ylabel("Suhu (F)")
plt.grid(True)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1)) # Place legend outside the plot area
plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to prevent legend overlap
plt.savefig("kurva_suhu_lengkap.png")
plt.close()

print("Grafik suhu lengkap berhasil dibuat.")

In [ ]:
wb = load_workbook(excel_file)
ws_full_temp = wb.create_sheet("Suhu Lengkap")
ws_full_temp.add_image(Image("kurva_suhu_lengkap.png"), "A1")
wb.save(excel_file)
print("Grafik suhu lengkap ditambahkan ke laporan Excel.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))
for logger in loggers:

    plt.plot(
        data["Waktu"],
        data[logger],
        label=logger
    )

plt.xlabel("Waktu")
plt.ylabel("Suhu (°F)")
plt.legend()
plt.grid()

plt.show()

In [ ]:
from google.colab import files

files.download('Laporan_Retort.xlsx')